<a href="https://colab.research.google.com/github/w2dw3d232wa/Lab3-/blob/main/Lab3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [9]:
import pandas as pd
from sklearn.model_selection import train_test_split

print("--- 終極破解版：直接從網路讀取資料 ---")

# 直接填入 GitHub 上的原始資料連結，Pandas 超級聰明，可以直接從網址讀取 CSV！
url = "https://raw.githubusercontent.com/sharmaroshan/Heart-UCI-Dataset/master/heart.csv"
df = pd.read_csv(url)

print("🎉 突破 Kaggle 封鎖！成功讀取資料！")
print("來看看前五筆病患的體檢報告：")
print(df.head())
print("-" * 30)

# 分離特徵 (X) 與標籤 (y)
# target: 1 代表有心臟病，0 代表健康
X = df.drop('target', axis=1)
y = df['target']

# 切分訓練集與測試集 (80% 訓練, 20% 測試)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"資料準備完成！準備送給 AI 訓練。")
print(f"訓練資料量: {len(X_train)} 筆")
print(f"測試資料量: {len(X_test)} 筆")

--- 終極破解版：直接從網路讀取資料 ---
🎉 突破 Kaggle 封鎖！成功讀取資料！
來看看前五筆病患的體檢報告：
   age  sex  cp  trestbps  chol  fbs  restecg  thalach  exang  oldpeak  slope  \
0   63    1   3       145   233    1        0      150      0      2.3      0   
1   37    1   2       130   250    0        1      187      0      3.5      0   
2   41    0   1       130   204    0        0      172      0      1.4      2   
3   56    1   1       120   236    0        1      178      0      0.8      2   
4   57    0   0       120   354    0        1      163      1      0.6      2   

   ca  thal  target  
0   0     1       1  
1   0     2       1  
2   0     2       1  
3   0     2       1  
4   0     2       1  
------------------------------
資料準備完成！準備送給 AI 訓練。
訓練資料量: 242 筆
測試資料量: 61 筆


In [11]:
import tensorflow_decision_forests as tfdf
import pandas as pd

print("--- 第三關：用 TensorFlow 種下三種樹狀分類器 (修正版) ---")

# 1. 轉換資料格式 (如果剛剛跑過了，這裡會再跑一次，沒關係)
train_df = pd.concat([X_train, y_train], axis=1)
test_df = pd.concat([X_test, y_test], axis=1)

train_ds = tfdf.keras.pd_dataframe_to_tf_dataset(train_df, label="target")
test_ds = tfdf.keras.pd_dataframe_to_tf_dataset(test_df, label="target")

# 2. 訓練第一種：基礎決策樹 (Decision Tree - CART)
print("\n🌲 正在訓練：基礎決策樹 (Decision Tree)...")
model_dt = tfdf.keras.CartModel(verbose=0)
model_dt.compile(metrics=["accuracy"]) # 老師剛剛漏掉了這張計分表！
model_dt.fit(train_ds)

# 3. 訓練第二種：隨機森林 (Random Forest)
print("🌲🌲🌲 正在訓練：隨機森林 (Random Forest)...")
model_rf = tfdf.keras.RandomForestModel(verbose=0)
model_rf.compile(metrics=["accuracy"]) # 補上計分表
model_rf.fit(train_ds)

# 4. 訓練第三種：梯度提升樹 (Gradient Boosted Trees)
print("🚀 正在訓練：梯度提升樹 (Boosted Trees)...")
model_gbt = tfdf.keras.GradientBoostedTreesModel(verbose=0)
model_gbt.compile(metrics=["accuracy"]) # 補上計分表
model_gbt.fit(train_ds)


print("\n--- 第四關：驗收期末考成績 ---")
# 讓三個模型去看那 61 筆沒看過的測試資料，看誰診斷心臟病最準！

def evaluate_model(model, name):
    evaluation = model.evaluate(test_ds, return_dict=True, verbose=0)
    print(f"[{name}] 測試準確率: {evaluation['accuracy']:.4f} (大約 {evaluation['accuracy']*100:.2f}%)")

evaluate_model(model_dt, "基礎決策樹")
evaluate_model(model_rf, "隨機森林")
evaluate_model(model_gbt, "梯度提升樹")

--- 第三關：用 TensorFlow 種下三種樹狀分類器 (修正版) ---

🌲 正在訓練：基礎決策樹 (Decision Tree)...
🌲🌲🌲 正在訓練：隨機森林 (Random Forest)...


🚀 正在訓練：梯度提升樹 (Boosted Trees)...



--- 第四關：驗收期末考成績 ---
[基礎決策樹] 測試準確率: 0.7541 (大約 75.41%)
[隨機森林] 測試準確率: 0.8361 (大約 83.61%)
[梯度提升樹] 測試準確率: 0.8361 (大約 83.61%)


訓練結束!!

# LAB 3：樹狀分類器心臟病預測 —— 實驗結果與分析報告

## 一、 實驗結果結論
藉由最終訓練成績可以得知：
* **基礎決策樹** 為最低的準確率（75.41%）。
* **隨機森林** 與 **梯度提升樹** 並列第一（83.61%）。

---

## 二、 模型表現深度分析

### 1. 基礎決策樹 (Decision Tree)
* **表現評估：** 準確率最低。
* **核心核心問題：** 單一決策樹極容易對訓練集產生**過度擬合 (Overfitting)**。這會導致模型流於「死背」現有資料的細節，一旦遇到全新未見過的測試資料時，泛化能力不足，導致準確率明顯下降。

### 2. 集成學習的強大證明 (Ensemble Learning)
其餘兩種模型則完美證明了機器學習中「集成學習」的強大效能：

* **隨機森林 (Random Forest)**
  * **核心概念：** 採用 **Bagging** 概念。
  * **運作機制：** 透過同時建立大量彼此獨立的決策樹，並在最後進行「多數決投票」。這種集體會診的方式，成功降低了單一模型的變異性（Variance），大幅提升了預測的穩定度與準確性。

* **梯度提升樹 (Gradient Boosted Trees)**
  * **核心概念：** 採用 **Boosting** 概念。
  * **運作機制：** 與隨機森林的平行投票不同，它是「循序漸進」的。讓每一棵新生成的樹，專門去修正前一棵樹所犯的誤差與缺點，透過不斷疊代與自我修正，能極度精準地抓出資料中複雜且高階的特徵。

---

## 三、 總結：樹狀分類器的兩大核心優勢
透過本次實驗，體現了樹狀分類器在處理醫療與結構化數據時，擁有比起傳統深度學習更適合的兩大優勢：

1. **高解釋能力 (Interpretability)**
   相比於深度神經網路（CNN/DNN）那種難以窺探內部邏輯的「黑盒子特性」，樹狀模型能夠清晰地追溯並畫出判斷的決策路徑（例如：年齡 $> 50$ 且膽固醇數值超標）。這在生醫研究或臨床輔助診斷上，能夠提供醫師高度信任且具備參考價值的決策理由。

2. **靈活的資料適應性**
   樹狀模型能直接應對混合了「數值型特徵」（如：年齡、血壓、膽固醇）與「類別型特徵」（如：性別、胸痛類型）的原始數據。在訓練前**不需要**像神經網路那樣進行繁瑣且嚴格的特徵縮放（Normalization），大幅簡化了資料前處理的複雜度與出錯率。